# 02 — Feature Engineering

## 1. Feature Engineering Strategy
- Time features (year, month, day)
- Customer features
- Product features
- Promotion features

## 2. Time Series Features
- Lag features
- Rolling mean
- Seasonality indicators

## 3. Data Merging
- Join all tables correctly
- Avoid leakage

## 4. Final Dataset
- Train/test split (time-based)

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

from src.data.load_data import load_sales
from src.data.preprocess import clean_sales
from src.features.build_features import build_supervised_frame, get_feature_columns

sales = clean_sales(load_sales(PROJECT_ROOT / 'data' / 'raw'))
sales.head()

In [ ]:
# Build supervised learning frame for Revenue
frame = build_supervised_frame(sales, target_col='Revenue')
feature_cols = get_feature_columns(frame, target_col='Revenue')
frame[feature_cols + ['Revenue']].head()

In [ ]:
# Time-based split example (last 365 days as validation)
val_days = 365
cutoff = frame['Date'].max() - pd.Timedelta(days=val_days)
train_frame = frame[frame['Date'] <= cutoff].dropna(subset=feature_cols + ['Revenue'])
val_frame = frame[frame['Date'] > cutoff].dropna(subset=feature_cols + ['Revenue'])
train_frame.shape, val_frame.shape

## Note on leakage
- For day $t$, lag/rolling features must be computed from days $e t-1$ only.
- For future dates (e.g., 2023+), only calendar + historical/predicted lags are guaranteed available in this dataset snapshot.